In [1]:
import arviz as az
import bambi as bmb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pymc as pm

print(f"Running on PyMC v{pm.__version__}")
import statsmodels.api as sm

from matplotlib.lines import Line2D

Running on PyMC v5.25.1


In [2]:
SEED = 1234
az.style.use("arviz-darkgrid")
plt.rcParams["figure.dpi"] = 300

In [3]:
# read the data
data = pd.read_csv('data/EAST_Data_Overview.csv')

data["SleepOrder"] = pd.to_numeric(data["SleepOrder"], errors="coerce")
data["SleepCondition"] = pd.to_numeric(data["SleepCondition"], errors="coerce")

# Recode: 1 -> -1, 2 -> 1
data["SleepOrder"] = data["SleepOrder"].replace({1: -1, 2: 1})
data["SleepCondition"] = data["SleepCondition"].replace({1: -1, 2: 1})

# print the first few rows of the data
print(data.head())
print(data.shape)

   VPN  SleepOrder  SleepCondition  pMisses  pPrematireResp  pNoSEva  nOKTr  \
0    1          -1               1      1.2             0.0      1.3    987   
1    2          -1               1      1.0             0.1      1.4    985   
2    3          -1               1      0.9             0.2      2.1    978   
3    4          -1               1      0.3             0.1      0.5    994   
4    6           1               1      2.2             0.0      3.3    967   

   nErrorall    pErrall  nArwareErr  ...  ESS  PVT_Standard  PVT_Standard_SD  \
0        179  18.135765          47  ...   16      540.5493        162.09916   
1         67   6.802030          40  ...   11      593.0541        217.96700   
2        186  19.018405          65  ...   15      521.9296         80.18698   
3        166  16.700201          49  ...    4      493.4177         43.80423   
4         99  10.237849          55  ...   16      495.6842         90.05749   

   PVT_novel  PVT_novel_SD  PVT_misses  Occ_

# Sleep questionnaires

In [4]:
model = bmb.Model("KSS ~ SleepCondition + SleepOrder + (1|VPN)", data)
idata = model.fit(draws=2000, chains=4, random_seed=SEED,mp_ctx='forkserver')
az.summary(idata, hdi_prob=0.95)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, SleepCondition, SleepOrder, 1|VPN_sigma, 1|VPN_offset]


/Users/hanskirschner/Desktop/EAST/MORPHEUS_EAST_Analyses/EAST_Data_and_Code/Bayesian_linear_models/.venv/lib/python
3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 9 seconds.
There were 40 divergences after tuning. Increase `target_accept` or reparameterize.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
sigma,1.421,0.187,1.077,1.791,0.005,0.002,1660.0,2623.0,1.0
Intercept,5.240,0.244,4.743,5.704,0.004,0.003,4508.0,3800.0,1.0
SleepCondition,1.507,0.184,1.145,1.873,0.002,0.003,8851.0,5730.0,1.0
SleepOrder,-0.326,0.183,-0.672,0.038,0.002,0.002,7184.0,5208.0,1.0
1|VPN_sigma,0.773,0.346,0.017,1.338,0.010,0.005,1146.0,2075.0,1.0
1|VPN[1],0.110,0.599,-1.091,1.383,0.007,0.009,7052.0,4308.0,1.0
1|VPN[2],0.091,0.605,-1.106,1.332,0.007,0.008,7054.0,4987.0,1.0
1|VPN[3],0.098,0.605,-1.164,1.296,0.008,0.008,6487.0,5091.0,1.0
1|VPN[4],0.097,0.596,-1.068,1.364,0.008,0.010,6216.0,5083.0,1.0
1|VPN[6],0.845,0.764,-0.351,2.441,0.016,0.009,2200.0,4197.0,1.0


In [5]:
model = bmb.Model("PVT_misses ~ SleepCondition + SleepOrder + (1|VPN)", data)
idata = model.fit(draws=2000, chains=4, random_seed=SEED,mp_ctx='forkserver')
az.summary(idata, hdi_prob=0.95)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, SleepCondition, SleepOrder, 1|VPN_sigma, 1|VPN_offset]


/Users/hanskirschner/Desktop/EAST/MORPHEUS_EAST_Analyses/EAST_Data_and_Code/Bayesian_linear_models/.venv/lib/python
3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 9 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
sigma,11.846,1.714,8.842,15.199,0.035,0.027,2446.0,3700.0,1.0
Intercept,44.073,3.648,36.737,51.163,0.087,0.052,1767.0,2544.0,1.0
SleepCondition,3.195,1.534,0.324,6.403,0.016,0.020,9094.0,5636.0,1.0
SleepOrder,2.414,1.550,-0.724,5.333,0.016,0.019,9717.0,6100.0,1.0
1|VPN_sigma,17.707,3.087,11.611,23.651,0.077,0.048,1610.0,2812.0,1.0
1|VPN[1],-1.708,7.999,-16.653,14.615,0.105,0.091,5827.0,5698.0,1.0
1|VPN[2],18.035,8.271,1.846,34.245,0.106,0.092,6058.0,5906.0,1.0
1|VPN[3],-8.532,7.985,-23.721,7.560,0.100,0.090,6304.0,6075.0,1.0
1|VPN[4],1.150,8.103,-15.230,16.659,0.106,0.094,5818.0,5267.0,1.0
1|VPN[6],-15.726,8.260,-32.644,0.084,0.112,0.109,5350.0,5163.0,1.0


# Error rate

In [6]:
model = bmb.Model("pErrall ~ SleepCondition + SleepOrder + (1|VPN)", data)
#idata = model.fit(draws=2000, chains=4, random_seed=SEED,cores=1)
idata = model.fit(draws=2000, chains=4, random_seed=SEED,mp_ctx='forkserver')
az.summary(idata, hdi_prob=0.95)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, SleepCondition, SleepOrder, 1|VPN_sigma, 1|VPN_offset]


/Users/hanskirschner/Desktop/EAST/MORPHEUS_EAST_Analyses/EAST_Data_and_Code/Bayesian_linear_models/.venv/lib/python
3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 9 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
sigma,3.694,0.542,2.760,4.800,0.011,0.010,2529.0,2577.0,1.0
Intercept,10.661,0.944,8.891,12.603,0.019,0.012,2480.0,3971.0,1.0
SleepCondition,1.547,0.481,0.581,2.465,0.005,0.006,10721.0,5736.0,1.0
SleepOrder,-0.710,0.490,-1.627,0.283,0.005,0.007,10548.0,5252.0,1.0
1|VPN_sigma,4.213,0.904,2.502,6.085,0.021,0.017,1982.0,2317.0,1.0
1|VPN[1],2.179,2.284,-2.301,6.629,0.029,0.028,6047.0,5010.0,1.0
1|VPN[2],-2.532,2.352,-6.939,2.266,0.029,0.025,6761.0,5999.0,1.0
1|VPN[3],0.639,2.279,-3.905,4.949,0.026,0.025,7488.0,5615.0,1.0
1|VPN[4],3.460,2.394,-1.097,8.223,0.031,0.026,5872.0,5776.0,1.0
1|VPN[6],-0.325,2.277,-4.883,3.993,0.025,0.027,8068.0,5732.0,1.0


In [7]:
model = bmb.Model("pAwareErr ~ SleepCondition + SleepOrder + (1|VPN)", data)
#idata = model.fit(draws=2000, chains=4, random_seed=SEED,cores=1)
idata = model.fit(draws=2000, chains=4, random_seed=SEED,mp_ctx='forkserver')
az.summary(idata, hdi_prob=0.95)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, SleepCondition, SleepOrder, 1|VPN_sigma, 1|VPN_offset]


/Users/hanskirschner/Desktop/EAST/MORPHEUS_EAST_Analyses/EAST_Data_and_Code/Bayesian_linear_models/.venv/lib/python
3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 8 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
sigma,9.042,1.297,6.653,11.562,0.025,0.016,2610.0,4225.0,1.0
Intercept,48.947,3.338,42.464,55.778,0.084,0.054,1578.0,2265.0,1.0
SleepCondition,-3.344,1.169,-5.592,-0.961,0.012,0.014,10122.0,6216.0,1.0
SleepOrder,-1.414,1.194,-3.679,1.023,0.012,0.016,9905.0,5565.0,1.0
1|VPN_sigma,16.551,2.676,11.712,22.101,0.067,0.046,1608.0,2460.0,1.0
1|VPN[1],-15.763,6.697,-28.937,-2.692,0.100,0.076,4518.0,5136.0,1.0
1|VPN[2],8.869,6.652,-3.795,22.118,0.102,0.072,4287.0,5679.0,1.0
1|VPN[3],-10.198,6.647,-22.976,3.092,0.097,0.076,4695.0,4859.0,1.0
1|VPN[4],-17.768,6.627,-30.926,-4.622,0.100,0.071,4432.0,5581.0,1.0
1|VPN[6],2.570,6.659,-10.406,15.458,0.092,0.073,5277.0,5751.0,1.0


In [8]:
model = bmb.Model("pMisses ~ SleepCondition + SleepOrder + (1|VPN)", data)
#idata = model.fit(draws=2000, chains=4, random_seed=SEED,cores=1)
idata = model.fit(draws=2000, chains=4, random_seed=SEED,mp_ctx='forkserver')
az.summary(idata, hdi_prob=0.95)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, SleepCondition, SleepOrder, 1|VPN_sigma, 1|VPN_offset]


/Users/hanskirschner/Desktop/EAST/MORPHEUS_EAST_Analyses/EAST_Data_and_Code/Bayesian_linear_models/.venv/lib/python
3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 9 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
sigma,2.144,0.282,1.617,2.690,0.005,0.003,2700.0,4478.0,1.0
Intercept,1.492,0.366,0.774,2.204,0.004,0.004,7188.0,5529.0,1.0
SleepCondition,0.885,0.285,0.311,1.433,0.002,0.005,14949.0,4949.0,1.0
SleepOrder,0.253,0.278,-0.322,0.788,0.003,0.004,11654.0,4876.0,1.0
1|VPN_sigma,1.134,0.515,0.019,1.966,0.014,0.007,1376.0,1888.0,1.0
1|VPN[1],-0.297,0.909,-2.211,1.482,0.010,0.013,8051.0,5768.0,1.0
1|VPN[2],-0.263,0.915,-2.221,1.405,0.009,0.012,9856.0,6411.0,1.0
1|VPN[3],-0.354,0.880,-2.140,1.341,0.010,0.011,8018.0,5916.0,1.0
1|VPN[4],-0.478,0.930,-2.421,1.298,0.012,0.012,6151.0,5187.0,1.0
1|VPN[6],-0.079,0.890,-1.762,1.824,0.009,0.013,10787.0,5963.0,1.0


In [9]:
model = bmb.Model("RT ~ SleepCondition + SleepOrder + (1|VPN)", data)
#idata = model.fit(draws=2000, chains=4, random_seed=SEED,cores=1)
idata = model.fit(draws=2000, chains=4, random_seed=SEED,mp_ctx='forkserver')
az.summary(idata, hdi_prob=0.95)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, SleepCondition, SleepOrder, 1|VPN_sigma, 1|VPN_offset]


/Users/hanskirschner/Desktop/EAST/MORPHEUS_EAST_Analyses/EAST_Data_and_Code/Bayesian_linear_models/.venv/lib/python
3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 9 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
sigma,0.045,0.006,0.033,0.057,0.000,0.0,2298.0,3132.0,1.0
Intercept,0.626,0.021,0.581,0.666,0.001,0.0,1322.0,2269.0,1.0
SleepCondition,0.011,0.006,-0.000,0.023,0.000,0.0,9966.0,5065.0,1.0
SleepOrder,-0.040,0.006,-0.052,-0.029,0.000,0.0,9722.0,5863.0,1.0
1|VPN_sigma,0.112,0.017,0.080,0.146,0.000,0.0,1382.0,1635.0,1.0
1|VPN[1],0.036,0.037,-0.036,0.107,0.001,0.0,3514.0,5096.0,1.0
1|VPN[2],-0.010,0.036,-0.084,0.057,0.001,0.0,3152.0,4438.0,1.0
1|VPN[3],0.006,0.036,-0.060,0.080,0.001,0.0,3604.0,4893.0,1.0
1|VPN[4],-0.087,0.037,-0.160,-0.016,0.001,0.0,3469.0,5024.0,1.0
1|VPN[6],-0.001,0.036,-0.073,0.068,0.001,0.0,3428.0,5114.0,1.0


In [10]:
data3 = df = pd.read_csv("data/EAST_Data_ErrD_Qs.csv", header=0)

# Recode: 1 -> -1, 2 -> 1
data3["SleepOrder"] = data3["SleepOrder"].replace({1: -1, 2: 1})
data3["SleepCond"] = data3["SleepCond"].replace({1: -1, 2: 1})
# print the first few rows of the data
print(data3.head())
print(data3.shape)

   SubID  SleepCond  SleepOrder     Q1_ErrD     Q2_ErrD     Q3_ErrD
0      1         -1          -1  342.324299  247.623971  287.157588
1      1          1          -1  577.761678  499.206486  563.135529
2      2         -1          -1  191.879571  180.955979  306.151249
3      2          1          -1  179.097789  173.186309  226.206502
4      3         -1          -1  152.059508   99.018813  114.136950
(60, 6)


In [11]:
model = bmb.Model("Q1_ErrD ~ SleepCond + SleepOrder + (1|SubID)", data3)
#idata = model.fit(draws=2000, chains=4, random_seed=SEED,cores=1)
idata = model.fit(draws=2000, chains=4, random_seed=SEED, mp_ctx='forkserver')
az.summary(idata, hdi_prob=0.95)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, SleepCond, SleepOrder, 1|SubID_sigma, 1|SubID_offset]


/Users/hanskirschner/Desktop/EAST/MORPHEUS_EAST_Analyses/EAST_Data_and_Code/Bayesian_linear_models/.venv/lib/python
3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 9 seconds.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
sigma,136.851,18.850,103.210,175.225,0.425,0.225,1986.0,3407.0,1.0
Intercept,323.111,25.532,274.528,374.863,0.355,0.360,5259.0,3860.0,1.0
SleepCond,8.438,17.815,-26.812,43.800,0.151,0.230,13863.0,5710.0,1.0
SleepOrder,-13.339,25.211,-63.396,37.468,0.364,0.299,4843.0,4705.0,1.0
1|SubID_sigma,90.979,34.962,16.268,158.250,0.984,0.616,1286.0,1276.0,1.0
1|SubID[1],57.350,70.504,-71.152,205.608,0.997,0.835,5157.0,4923.0,1.0
1|SubID[2],-69.462,73.635,-219.737,59.181,1.095,0.788,4478.0,6028.0,1.0
1|SubID[3],11.575,65.043,-113.057,149.119,0.709,0.871,8419.0,6125.0,1.0
1|SubID[4],54.166,68.953,-68.381,200.919,0.981,0.753,5025.0,5695.0,1.0
1|SubID[5],10.166,66.953,-123.402,143.033,0.768,0.861,7691.0,5541.0,1.0


In [12]:
model = bmb.Model("Q2_ErrD ~ SleepCond + SleepOrder + (1|SubID)", data3)
#idata = model.fit(draws=2000, chains=4, random_seed=SEED,cores=1)
idata = model.fit(draws=2000, chains=4, random_seed=SEED, mp_ctx='forkserver')
az.summary(idata, hdi_prob=0.95)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, SleepCond, SleepOrder, 1|SubID_sigma, 1|SubID_offset]


/Users/hanskirschner/Desktop/EAST/MORPHEUS_EAST_Analyses/EAST_Data_and_Code/Bayesian_linear_models/.venv/lib/python
3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 10 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
sigma,116.365,16.782,86.003,150.342,0.454,0.345,1726.0,2356.0,1.0
Intercept,291.033,28.963,232.440,347.261,0.547,0.368,2806.0,4051.0,1.0
SleepCond,44.261,14.854,15.891,75.009,0.149,0.195,9879.0,5204.0,1.0
SleepOrder,-15.104,28.673,-70.040,43.606,0.550,0.387,2719.0,3780.0,1.0
1|SubID_sigma,129.354,28.886,71.755,185.076,0.767,0.650,1527.0,1998.0,1.0
1|SubID[1],47.186,72.479,-93.818,189.563,0.899,0.758,6556.0,6118.0,1.0
1|SubID[2],-88.796,76.596,-241.846,55.690,1.048,0.814,5333.0,5519.0,1.0
1|SubID[3],6.336,71.777,-137.017,144.712,0.905,0.827,6261.0,5845.0,1.0
1|SubID[4],108.497,76.840,-36.727,260.497,1.075,0.831,5097.0,5587.0,1.0
1|SubID[5],5.718,74.144,-141.686,150.952,0.963,0.903,5950.0,4517.0,1.0


In [13]:
model = bmb.Model("Q3_ErrD ~ SleepCond + SleepOrder + (1|SubID)", data3)
#idata = model.fit(draws=2000, chains=4, random_seed=SEED,cores=1)
idata = model.fit(draws=2000, chains=4, random_seed=SEED, mp_ctx='forkserver')
az.summary(idata, hdi_prob=0.95)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, SleepCond, SleepOrder, 1|SubID_sigma, 1|SubID_offset]


/Users/hanskirschner/Desktop/EAST/MORPHEUS_EAST_Analyses/EAST_Data_and_Code/Bayesian_linear_models/.venv/lib/python
3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 9 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
sigma,136.960,19.579,103.326,178.398,0.491,0.283,1663.0,3122.0,1.0
Intercept,328.982,28.595,270.872,382.977,0.499,0.386,3313.0,3929.0,1.0
SleepCond,70.334,18.216,32.913,104.443,0.188,0.237,9349.0,5360.0,1.0
SleepOrder,-22.563,28.811,-79.131,33.507,0.482,0.373,3613.0,3904.0,1.0
1|SubID_sigma,116.479,33.886,42.193,179.653,0.925,0.621,1385.0,1738.0,1.0
1|SubID[1],41.557,76.887,-102.791,198.664,1.020,0.955,5709.0,4814.0,1.0
1|SubID[2],-49.719,77.918,-206.124,99.078,1.054,0.889,5518.0,4801.0,1.0
1|SubID[3],2.856,74.419,-135.689,161.366,0.908,0.948,6643.0,5498.0,1.0
1|SubID[4],105.379,82.592,-49.341,271.397,1.423,0.941,3382.0,4985.0,1.0
1|SubID[5],-19.229,76.968,-174.010,131.353,0.932,0.963,6823.0,5783.0,1.0


In [14]:
data4 = df = pd.read_csv("data/EAST_Data_UnAwErrD_Qs.csv", header=0)

# Recode: 1 -> -1, 2 -> 1
data4["SleepOrder"] = data4["SleepOrder"].replace({1: -1, 2: 1})
data4["SleepCond"] = data4["SleepCond"].replace({1: -1, 2: 1})
# print the first few rows of the data
print(data4.head())
print(data4.shape)

   SubID  SleepCond  SleepOrder  Q1_UnAwErrD  Q2_UnAwErrD  Q3_UnAwErrD
0      1         -1          -1   247.623971   152.383982   180.955979
1      1          1          -1   364.755278   384.918500   448.847543
2      2         -1          -1   115.687580    57.143993    96.639582
3      2          1          -1    82.808803    76.208299    85.715990
4      3         -1          -1    75.567117    70.446817    76.040955
(60, 6)


In [15]:
model = bmb.Model("Q1_UnAwErrD ~ SleepCond + SleepOrder + (1|SubID)", data4)
#idata = model.fit(draws=2000, chains=4, random_seed=SEED,cores=1)
idata = model.fit(draws=2000, chains=4, random_seed=SEED, mp_ctx='forkserver')
az.summary(idata, hdi_prob=0.95)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, SleepCond, SleepOrder, 1|SubID_sigma, 1|SubID_offset]


/Users/hanskirschner/Desktop/EAST/MORPHEUS_EAST_Analyses/EAST_Data_and_Code/Bayesian_linear_models/.venv/lib/python
3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 8 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
sigma,85.607,12.146,64.523,110.942,0.266,0.213,2249.0,2824.0,1.0
Intercept,178.424,21.486,136.590,221.059,0.444,0.317,2359.0,3517.0,1.0
SleepCond,5.356,11.191,-16.431,27.720,0.104,0.146,11577.0,5794.0,1.0
SleepOrder,-35.269,20.912,-77.020,5.266,0.420,0.272,2484.0,3712.0,1.0
1|SubID_sigma,94.645,21.305,55.223,137.695,0.557,0.430,1515.0,2334.0,1.0
1|SubID[1],63.943,55.025,-43.405,172.099,0.760,0.602,5252.0,5704.0,1.0
1|SubID[2],-80.280,56.549,-190.651,31.147,0.805,0.632,4943.0,5338.0,1.0
1|SubID[3],-18.194,54.091,-123.045,87.245,0.727,0.580,5521.0,5466.0,1.0
1|SubID[4],86.564,55.991,-23.450,196.580,0.885,0.632,3998.0,5255.0,1.0
1|SubID[5],16.176,54.572,-89.794,124.612,0.693,0.623,6224.0,5341.0,1.0


In [16]:
model = bmb.Model("Q2_UnAwErrD ~ SleepCond + SleepOrder + (1|SubID)", data4)
#idata = model.fit(draws=2000, chains=4, random_seed=SEED,cores=1)
idata = model.fit(draws=2000, chains=4, random_seed=SEED, mp_ctx='forkserver')
az.summary(idata, hdi_prob=0.95)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, SleepCond, SleepOrder, 1|SubID_sigma, 1|SubID_offset]


/Users/hanskirschner/Desktop/EAST/MORPHEUS_EAST_Analyses/EAST_Data_and_Code/Bayesian_linear_models/.venv/lib/python
3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 9 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
sigma,94.235,13.234,69.384,119.819,0.271,0.192,2526.0,3604.0,1.0
Intercept,162.230,25.480,111.304,211.539,0.532,0.344,2312.0,3275.0,1.0
SleepCond,34.249,12.257,10.336,58.966,0.121,0.162,10226.0,5514.0,1.0
SleepOrder,-18.162,25.513,-68.759,32.271,0.557,0.347,2104.0,3454.0,1.0
1|SubID_sigma,119.684,23.771,72.904,166.815,0.618,0.361,1489.0,3031.0,1.0
1|SubID[1],65.653,64.318,-61.539,192.696,0.915,0.720,4933.0,5472.0,1.0
1|SubID[2],-84.835,66.336,-214.957,46.112,0.976,0.721,4638.0,5417.0,1.0
1|SubID[3],31.508,62.808,-94.983,151.334,0.894,0.720,4955.0,5633.0,1.0
1|SubID[4],98.140,66.231,-27.735,232.166,0.979,0.715,4592.0,5401.0,1.0
1|SubID[5],-3.771,63.934,-132.013,122.855,0.860,0.694,5514.0,5923.0,1.0


In [17]:
model = bmb.Model("Q3_UnAwErrD ~ SleepCond + SleepOrder + (1|SubID)", data4)
#idata = model.fit(draws=2000, chains=4, random_seed=SEED,cores=1)
idata = model.fit(draws=2000, chains=4, random_seed=SEED, mp_ctx='forkserver')
az.summary(idata, hdi_prob=0.95)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, SleepCond, SleepOrder, 1|SubID_sigma, 1|SubID_offset]


/Users/hanskirschner/Desktop/EAST/MORPHEUS_EAST_Analyses/EAST_Data_and_Code/Bayesian_linear_models/.venv/lib/python
3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 9 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
sigma,97.830,13.683,72.759,124.497,0.257,0.179,2808.0,4731.0,1.0
Intercept,188.847,29.303,127.864,242.791,0.593,0.417,2448.0,3387.0,1.0
SleepCond,53.826,12.655,30.140,80.202,0.132,0.154,9194.0,5599.0,1.0
SleepOrder,-25.742,28.433,-83.273,29.738,0.562,0.399,2566.0,3296.0,1.0
1|SubID_sigma,137.234,25.474,89.120,188.073,0.588,0.399,1912.0,3131.0,1.0
1|SubID[1],78.215,69.613,-63.213,209.808,0.967,0.745,5201.0,5311.0,1.0
1|SubID[2],-95.973,70.731,-237.856,41.682,0.966,0.790,5357.0,5472.0,1.0
1|SubID[3],30.229,68.883,-104.764,167.672,0.945,0.806,5312.0,5102.0,1.0
1|SubID[4],132.425,70.459,-7.332,270.832,0.973,0.754,5254.0,5873.0,1.0
1|SubID[5],-46.565,69.749,-182.389,87.702,0.942,0.823,5466.0,5618.0,1.0
